In [1]:
# ==========================================
# 1️⃣ Import Required Libraries
# ==========================================

import numpy as np
import pandas as pd
import os

# Sklearn - preprocessing & modeling
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
import sklearn
print(sklearn.__version__)

1.6.1


# Dataset Used :
/kaggle/input/datasets/orvile/health-and-sleep-relation-2024

In [3]:
# ==========================================
# 2️⃣ Load Dataset
# ==========================================

df = pd.read_csv(
    "/kaggle/input/datasets/orvile/health-and-sleep-relation-2024/Health and Sleep relation 2024/Sleep_health_and_lifestyle_dataset.csv"
)

# Display first 5 rows
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


In [4]:
df = df.drop_duplicates(subset=df.columns.drop("Person ID"))

In [5]:
df.shape

(132, 13)

In [6]:
# ==========================================
# 3️⃣ Drop Unnecessary Columns
# ==========================================

# Remove ID column (not useful for prediction)
df = df.drop('Person ID', axis=1)

# Remove Sleep Disorder (we are predicting Quality of Sleep only)
df = df.drop('Sleep Disorder', axis=1)

df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200
1,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000
3,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000
5,Male,28,Software Engineer,5.9,4,30,8,Obese,140/90,85,3000
6,Male,29,Teacher,6.3,6,40,7,Obese,140/90,82,3500


In [7]:
# ==========================================
# 4️⃣ Reduce Rare Categories in Occupation
# ==========================================

# If any occupation appears less than 10 times,
# group it into "other" to avoid too many one-hot columns.

df['Occupation'] = df['Occupation'].apply(
    lambda x: 'other' if df['Occupation'].value_counts()[x] < 10 else x
)

df['Occupation'].value_counts()

Occupation
Nurse         29
Doctor        24
Engineer      22
other         16
Teacher       15
Lawyer        15
Accountant    11
Name: count, dtype: int64

In [18]:
df['BMI Category'].value_counts()

BMI Category
Normal           57
Overweight       52
Normal Weight    16
Obese             7
Name: count, dtype: int64

In [21]:
df['Physical Activity Level'].sort_values(ascending=False).head(1)

280    90
Name: Physical Activity Level, dtype: int64

In [8]:
# ==========================================
# 5️⃣ Split Blood Pressure into Two Columns
# ==========================================

# Example: "120/80" → Systolic = 120, Diastolic = 80

df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True)

df['Systolic_BP'] = df['Systolic_BP'].astype(int)
df['Diastolic_BP'] = df['Diastolic_BP'].astype(int)

# Drop original column
df = df.drop('Blood Pressure', axis=1)

df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Systolic_BP,Diastolic_BP
0,Male,27,other,6.1,6,42,6,Overweight,77,4200,126,83
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,125,80
3,Male,28,other,5.9,4,30,8,Obese,85,3000,140,90
5,Male,28,other,5.9,4,30,8,Obese,85,3000,140,90
6,Male,29,Teacher,6.3,6,40,7,Obese,82,3500,140,90


In [9]:
# ==========================================
# 6️⃣ Define X (features) and y (target)
# ==========================================

X = df.drop('Quality of Sleep', axis=1)
y = df['Quality of Sleep']

In [10]:
# ==========================================
# 7️⃣ Split Data into Train and Test
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [11]:
# ==========================================
# 8️⃣ Define Categorical and Numerical Columns
# ==========================================

categorical_cols = ['Gender', 'Occupation', 'BMI Category']

numeric_cols = [
    'Age',
    'Sleep Duration',
    'Physical Activity Level',
    'Stress Level',
    'Heart Rate',
    'Daily Steps',
    'Systolic_BP',
    'Diastolic_BP'
]

In [12]:
# ==========================================
# 9️⃣ Create Column Transformer
# ==========================================

# Numerical → pass as it is
# Categorical → One Hot Encoding

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [13]:
# ==========================================
# 🔟 Define Models
# ==========================================

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42)
}

In [14]:
# ==========================================
# 1️⃣1️⃣ Compare Models using Cross Validation
# ==========================================

for name, model in models.items():
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    
    rmse = -np.mean(scores)
    
    print(f"Model: {name} | RMSE: {rmse:.4f}")

Model: Linear Regression | RMSE: 0.4211
Model: Decision Tree | RMSE: 0.5219
Model: Random Forest | RMSE: 0.3537


In [15]:
# ==========================================
# 1️⃣2️⃣ Train Final Model
# ==========================================

best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

best_pipeline.fit(X_train, y_train)

pred = best_pipeline.predict(X_test)

In [16]:
# ==========================================
# 1️⃣3️⃣ Evaluate Model Performance
# ==========================================

train_score = best_pipeline.score(X_train, y_train)  # R²
test_score = best_pipeline.score(X_test, y_test)     # R²

print("Train R2:", train_score)
print("Test R2:", test_score)

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("Test RMSE:", rmse)

Train R2: 0.9882633228840125
Test R2: 0.9438286885245901
Test RMSE: 0.2908671581731147


In [17]:
# ==========================================
# 1️⃣4️⃣ Correlation with Target
# ==========================================

df.corr(numeric_only=True)['Quality of Sleep']\
    .sort_values(ascending=False)

Quality of Sleep           1.000000
Sleep Duration             0.869302
Age                        0.488256
Physical Activity Level    0.289501
Daily Steps                0.147607
Diastolic_BP              -0.155859
Systolic_BP               -0.180076
Heart Rate                -0.660056
Stress Level              -0.883948
Name: Quality of Sleep, dtype: float64

In [ ]:
import joblib
joblib.dump(best_pipeline,"sleep_quality_model.pkl")

## 📊 Average Sleep Duration by Gender

We calculated the mean sleep duration grouped by gender to analyze sleep pattern differences.

### 🔎 Result
- Female: **7.23 hours**
- Male: **7.04 hours**

### 📌 Insight
The difference is approximately **0.19 hours (~11 minutes)**, which is relatively small. This suggests there is no major gender-based difference in sleep duration within this dataset.

In [ ]:
avg_sleep=df.groupby('Gender')['Sleep Duration'].mean()
avg_sleep

## 📊 Average Quality of Sleep by Gender

We calculated the mean **Quality of Sleep** grouped by gender to analyze differences in perceived sleep quality.

### 🔎 Result
- Female: **7.44**
- Male: **6.86**

### 📌 Insight
Females report a higher average quality of sleep compared to males,with approx small difference This indicates a noticeable gender-based variation in sleep quality within this dataset.

## 📊 Gender-wise Sleep quality Analysis

grouped the dataset by **Gender** and calculated:
- Mean
- Median
- Standard Deviation
- Count



### 📌 Insight

- Female participants have a **higher average sleep score** compared to males.
- Median values confirm this trend (8 vs 7).
- Female scores show slightly **more variability** (higher standard deviation).
- The sample sizes are nearly balanced, so comparison is reliable.

➡️ Overall, gender appears to have a noticeable relationship with sleep score.

In [ ]:
quality_of_sleep=df.groupby('Gender')['Quality of Sleep'].agg(['mean','median','std','count'])
quality_of_sleep

In [ ]:
import joblib
import pandas as pd

# Load saved pipeline
model = joblib.load("sleep_quality_model.pkl")

def predict():
    
    gender = input("Enter gender (Male/Female): ").title()
    age = int(input("Enter age: "))
    occupation = input("Enter occupation: ").title()
    sleep_duration = float(input("Enter sleep duration (hours): "))
    physical_activity = int(input("Enter physical activity level: "))
    stress = int(input("Enter stress level: "))
    bmi = input("Enter BMI Category (Normal/Overweight/Obese): ").title()
    heart_rate = int(input("Enter heart rate: "))
    daily_steps = int(input("Enter daily steps: "))
    systolic = int(input("Enter systolic BP: "))
    diastolic = int(input("Enter diastolic BP: "))
    
    # Create dataframe with EXACT column names
    input_data = pd.DataFrame([{
        "Gender": gender,
        "Age": age,
        "Occupation": occupation,
        "Sleep Duration": sleep_duration,
        "Physical Activity Level": physical_activity,
        "Stress Level": stress,
        "BMI Category": bmi,
        "Heart Rate": heart_rate,
        "Daily Steps": daily_steps,
        "Systolic_BP": systolic,
        "Diastolic_BP": diastolic
    }])


    prediction=model.predict(input_data)
    print("quality of sleep (1-10)\n 1:bad quality sleep \n 10:good quality sleep \n",round(prediction[0],2))

In [ ]:
#predict()